In [1]:
import pandas as pd
import numpy as np
import csv
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2007_2008"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HIQ_E.xpt"
csv_file = DATA_DIR / "HIQ_E_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for fidelity comparison
source_df = df.copy()


print("=" * 80)
print("NHANES 2007-2008 HIQ_E XPT -> CSV")
print("=" * 80)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HIQ011",
    "HIQ031A",
    "HIQ031B",
    "HIQ031C",
    "HIQ031D",
    "HIQ031E",
    "HIQ031F",
    "HIQ031G",
    "HIQ031H",
    "HIQ031I",
    "HIQ031J",
    "HIQ031AA",
    "HIQ260",
    "HIQ105",
    "HIQ270",
    "HIQ210"
]


if df.shape != (10149, 17):
    raise ValueError(
        f"Unexpected HIQ_E shape: {df.shape}"
    )


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "HIQ_E columns/order differ.\n"
        f"Actual: {df.columns.tolist()}"
    )


if df.columns.duplicated().any():
    raise ValueError(
        "Duplicate column names found."
    )


print(
    "PASS: HIQ_E structure = "
    "10,149 rows x 17 columns."
)


# ============================================================
# 4. CDC CHECK ITEMS
# ============================================================

#
# CDC HTML contains:
#
# HIQ065 - CHECK ITEM
# HIQ259 - CHECK ITEM
# HIQ269 - CHECK ITEM
#
# These are routing instructions.
# They are NOT released variables in HIQ_E.xpt.
#
# DO NOT add them to the CSV.
#

CHECK_ITEMS = [
    "HIQ065",
    "HIQ259",
    "HIQ269"
]


for col in CHECK_ITEMS:

    if col in df.columns:
        raise ValueError(
            f"Unexpected CHECK ITEM in released XPT: {col}"
        )


print(
    "PASS: HIQ065, HIQ259 and HIQ269 "
    "correctly absent from released XPT."
)


# ============================================================
# 5. FIELD CLASSIFICATION
# ============================================================

#
# Actual XPT inspection:
#
# Genuine decimal fields = 0
#
# All released HIQ_E variables are:
#
# - identifier
# - questionnaire codes
#
# Therefore all can be clean Int64 after we verify
# that every nonmissing source value is whole-number.
#

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()

DECIMAL_COLUMNS = []


print(
    "\nInteger/code fields:",
    len(INTEGER_COLUMNS)
)

print(
    "Genuine decimal fields:",
    DECIMAL_COLUMNS
)


# ============================================================
# 6. KNOWN SAS/XPORT TINY VALUE CHECK
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = df.select_dtypes(
    include=[np.number]
).columns.tolist()


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print(
    "\n--- ORIGINAL XPT TINY VALUE CHECK ---"
)

print(
    "Tiny artifact occurrences:",
    tiny_total
)


if tiny_total != 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifact found. "
        "Do NOT automatically replace it with zero."
    )


print(
    "PASS: HIQ_E contains NO tiny XPORT artifacts."
)


# ============================================================
# 7. TRUE NUMERIC ZERO CHECK
# ============================================================

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print(
    "\nTrue source numeric zeros:",
    true_zero_total
)


if true_zero_total != 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )

    raise ValueError(
        "Unexpected true source zero(s) found."
    )


print(
    "PASS: Original HIQ_E contains no numeric zeros."
)


# ============================================================
# 8. SEQN VALIDATION
# ============================================================

seqn = df["SEQN"]


if int(
    seqn.isna().sum()
) != 0:
    raise ValueError(
        "Missing SEQN."
    )


if int(
    seqn.duplicated().sum()
) != 0:
    raise ValueError(
        "Duplicate SEQN."
    )


if seqn.min() != 41475:
    raise ValueError(
        f"Unexpected minimum SEQN: {seqn.min()}"
    )


if seqn.max() != 51623:
    raise ValueError(
        f"Unexpected maximum SEQN: {seqn.max()}"
    )


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=0
    )
):
    raise ValueError(
        "SEQN contains fractional values."
    )


print(
    "\nPASS: SEQN range = 41475-51623."
)

print(
    "PASS: Missing SEQN = 0."
)

print(
    "PASS: Duplicate SEQN = 0."
)


# ============================================================
# 9. FRACTIONAL / DECIMAL AUDIT
# ============================================================

fractional_columns = {}


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    fractional_count = int(
        (
            ~np.isclose(
                values,
                np.round(values),
                rtol=0,
                atol=1e-12
            )
        ).sum()
    )


    if fractional_count > 0:
        fractional_columns[
            col
        ] = fractional_count


print(
    "\n--- FRACTIONAL VALUE CHECK ---"
)

print(
    "Genuine fractional variables:",
    fractional_columns
)


if fractional_columns:
    raise ValueError(
        "STOP: Unexpected genuine decimal values found:\n"
        + str(
            fractional_columns
        )
    )


print(
    "PASS: Genuine decimal variables = 0."
)


# ============================================================
# 10. CDC / XPT FREQUENCY EXPECTATIONS
# ============================================================

#
# These counts were verified against:
#
# 1. Attached HIQ_E.xpt
# 2. Official CDC 2007-2008 HIQ_E dictionary
#

EXPECTED_FREQUENCIES = {

    "HIQ011": {
        1: 8145,
        2: 1983,
        7: 7,
        9: 14
    },

    "HIQ031A": {
        14: 4880,
        77: 3,
        99: 18
    },

    "HIQ031B": {
        15: 1506
    },

    "HIQ031C": {
        16: 37
    },

    "HIQ031D": {
        17: 1694
    },

    "HIQ031E": {
        18: 125
    },

    "HIQ031F": {
        19: 217
    },

    "HIQ031G": {
        20: 2
    },

    "HIQ031H": {
        21: 452
    },

    "HIQ031I": {
        22: 311
    },

    "HIQ031J": {
        23: 228
    },

    "HIQ031AA": {
        40: 4
    },

    "HIQ260": {
        1: 148,
        2: 90,
        7: 1,
        9: 9
    },

    "HIQ105": {
        1: 1070,
        2: 111
    },

    "HIQ270": {
        1: 7570,
        2: 509,
        7: 2,
        9: 80
    },

    "HIQ210": {
        1: 593,
        2: 7563,
        9: 2
    }
}


EXPECTED_MISSING = {

    "HIQ011": 0,

    "HIQ031A": 5248,
    "HIQ031B": 8643,
    "HIQ031C": 10112,
    "HIQ031D": 8455,
    "HIQ031E": 10024,
    "HIQ031F": 9932,
    "HIQ031G": 10147,
    "HIQ031H": 9697,
    "HIQ031I": 9838,
    "HIQ031J": 9921,
    "HIQ031AA": 10145,

    "HIQ260": 9901,
    "HIQ105": 8968,
    "HIQ270": 1988,
    "HIQ210": 1991
}


# ============================================================
# 11. CHECK CDC FREQUENCIES
# ============================================================

print(
    "\n--- CDC FREQUENCY CHECK ---"
)


for col, expected_counts in (
    EXPECTED_FREQUENCIES.items()
):

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"{col} frequency mismatch.\n"
            f"Expected: {expected_counts}\n"
            f"Actual:   {actual_counts}"
        )


    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    expected_missing = (
        EXPECTED_MISSING[col]
    )


    if actual_missing != expected_missing:

        raise ValueError(
            f"{col} missing count mismatch.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )


    print(
        f"PASS: {col}"
    )


# ============================================================
# 12. IMPORTANT HIQ260 PUBLIC-RELEASE NOTE
# ============================================================

#
# CDC states:
#
# One participant originally had a response to HIQ260,
# but it was changed to missing during data processing
# because that person should not have answered HIQ260.
#
# IMPORTANT:
#
# The public XPT already contains the corrected released
# value.
#
# DO NOT attempt to restore/reconstruct the old response.
#

if int(
    df["HIQ260"]
    .notna()
    .sum()
) != 248:

    raise ValueError(
        "HIQ260 nonmissing count differs."
    )


if int(
    df["HIQ260"]
    .isna()
    .sum()
) != 9901:

    raise ValueError(
        "HIQ260 missing count differs."
    )


print(
    "\nPASS: HIQ260 public-release distribution retained."
)

print(
    "      Valid   = 248"
)

print(
    "      Missing = 9,901"
)


# ============================================================
# 13. EXPECTED DATASET FOR FIDELITY
# ============================================================

#
# HIQ_E requires NO source-value correction.
#
# Tiny artifacts = 0
# Zero restorations = 0
# Genuine decimal variables = 0
#
# Therefore expected_df is EXACTLY the released XPT.
#

expected_df = source_df.copy()


# ============================================================
# 14. VERIFY ALL INTEGER FIELDS ARE WHOLE
# ============================================================

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy()
    )


    if len(values) == 0:
        continue


    if not np.all(
        np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=0
        )
    ):

        raise ValueError(
            f"STOP: {col} contains a non-whole value."
        )


print(
    "\nPASS: Every released HIQ_E field "
    "contains whole-number source values."
)


# ============================================================
# 15. CLEAN INTEGER FORMATTING
# ============================================================

#
# Convert:
#
# SEQN:
# 41475.0 -> 41475
#
# HIQ011:
# 1.0 -> 1
#
# HIQ031A:
# 14.0 -> 14
#
# HIQ270:
# 9.0 -> 9
#
# Missing stays blank at export.
#

for col in INTEGER_COLUMNS:

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "PASS: All HIQ_E variables converted "
    "to clean integer-style formatting."
)


# ============================================================
# 16. PRE-EXPORT EXACT NUMERIC FIDELITY
# ============================================================

pre_export_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    equal = np.isclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not bool(
        np.all(equal)
    ):

        pre_export_failures.append(
            col
        )


if pre_export_failures:

    raise ValueError(
        "Unexpected pre-export numeric changes:\n"
        + str(
            pre_export_failures
        )
    )


print(
    "PASS: Pre-export values equal original XPT exactly."
)


# ============================================================
# 17. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# NO:
#
# - round()
# - float_format
# - arbitrary recoding
# - missing-value filling
#
# Missing XPT values -> blank CSV cells.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print(
    "\nCSV created:"
)

print(
    csv_file
)


# ============================================================
# 18. RE-READ CSV WITH ROUND-TRIP PARSING
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 19. STRUCTURE AFTER EXPORT
# ============================================================

if csv_df.shape != expected_df.shape:

    raise ValueError(
        "CSV shape differs from XPT.\n"
        f"Expected: {expected_df.shape}\n"
        f"Actual:   {csv_df.shape}"
    )


if csv_df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "\nPASS: CSV structure matches XPT."
)


# ============================================================
# 20. EXACT XPT -> CSV NUMERIC FIDELITY
# ============================================================

numeric_failures = []
total_numeric_differences = 0


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )


    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    equal = np.isclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    differences = int(
        (~equal).sum()
    )


    total_numeric_differences += (
        differences
    )


    if differences > 0:

        numeric_failures.append(
            (
                col,
                differences
            )
        )


print(
    "XPT -> CSV numeric differences:",
    total_numeric_differences
)


if numeric_failures:

    raise ValueError(
        "Numeric fidelity failure:\n"
        + str(
            numeric_failures
        )
    )


print(
    "PASS: All 17 columns numerically "
    "match original XPT exactly."
)


# ============================================================
# 21. MISSING-POSITION FIDELITY
# ============================================================

missing_mismatches = 0
missing_failures = []


for col in EXPECTED_COLUMNS:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    differences = int(
        (
            expected_missing
            !=
            exported_missing
        ).sum()
    )


    missing_mismatches += (
        differences
    )


    if differences > 0:

        missing_failures.append(
            (
                col,
                differences
            )
        )


print(
    "Missing-position differences:",
    missing_mismatches
)


if missing_failures:

    raise ValueError(
        "Missing-position fidelity failure:\n"
        + str(
            missing_failures
        )
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 22. READ RAW CSV TOKENS
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8",
    newline=""
) as f:

    raw_rows = list(
        csv.DictReader(f)
    )


if len(raw_rows) != 10149:

    raise ValueError(
        "Unexpected raw CSV row count."
    )


# ============================================================
# 23. STRICT RAW TOKEN FIDELITY
# ============================================================

#
# Every nonmissing HIQ_E value should be:
#
# str(int(source_value))
#
# Examples:
#
# SEQN:
# 41475.0 -> "41475"
#
# HIQ031A:
# 14.0 -> "14"
#
# HIQ260:
# 9.0 -> "9"
#
# Missing:
# NaN -> ""
#

raw_mismatches = 0
raw_examples = []


for i, row in enumerate(
    raw_rows
):

    for col in EXPECTED_COLUMNS:

        source_value = (
            expected_df.iloc[i][col]
        )


        expected_text = (
            ""
            if pd.isna(source_value)
            else str(
                int(source_value)
            )
        )


        actual_text = (
            row[col]
        )


        if actual_text != expected_text:

            raw_mismatches += 1


            if len(
                raw_examples
            ) < 20:

                raw_examples.append(
                    {
                        "csv_row": i + 2,
                        "column": col,
                        "expected": expected_text,
                        "actual": actual_text
                    }
                )


print(
    "\nRaw-format differences:",
    raw_mismatches
)


if raw_mismatches != 0:

    print(
        "\nExamples:"
    )

    for example in raw_examples:
        print(example)


    raise ValueError(
        "HIQ_E raw CSV formatting differs."
    )


print(
    "PASS: Every raw CSV token matches "
    "the expected source representation."
)


# ============================================================
# 24. UNWANTED .0 CHECK
# ============================================================

dot_zero_count = 0
dot_zero_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col in INTEGER_COLUMNS:

        value = row[col]


        if (
            value != ""
            and
            value.endswith(".0")
        ):

            dot_zero_count += 1


            if len(
                dot_zero_examples
            ) < 20:

                dot_zero_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Unwanted '.0' values:",
    dot_zero_count
)


if dot_zero_count != 0:

    print(
        dot_zero_examples
    )

    raise ValueError(
        "Unwanted .0 formatting remains."
    )


print(
    "PASS: No unnecessary .0 values."
)


# ============================================================
# 25. SCIENTIFIC NOTATION CHECK
# ============================================================

scientific_count = 0
scientific_examples = []


for row_number, row in enumerate(
    raw_rows,
    start=2
):

    for col, value in row.items():

        lower = value.lower()


        if (
            "e-" in lower
            or
            "e+" in lower
        ):

            scientific_count += 1


            if len(
                scientific_examples
            ) < 20:

                scientific_examples.append(
                    (
                        row_number,
                        col,
                        value
                    )
                )


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    print(
        scientific_examples
    )

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 26. TINY ARTIFACT STRING CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


tiny_string_count = (
    csv_text.count(
        "5.397605346934028e-79"
    )
)


print(
    "Tiny artifact strings:",
    tiny_string_count
)


if tiny_string_count != 0:

    raise ValueError(
        "Tiny XPORT artifact found in CSV."
    )


print(
    "PASS: No tiny artifact in CSV."
)


# ============================================================
# 27. FINAL FREQUENCY RECHECK
# ============================================================

print(
    "\n--- FINAL CDC/XPT FREQUENCY RECHECK ---"
)


for col, expected_counts in (
    EXPECTED_FREQUENCIES.items()
):

    actual_counts = (
        csv_df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )


    actual_counts = {
        int(k): int(v)
        for k, v in actual_counts.items()
    }


    if actual_counts != expected_counts:

        raise ValueError(
            f"Final {col} frequency changed."
        )


    if int(
        csv_df[col]
        .isna()
        .sum()
    ) != EXPECTED_MISSING[col]:

        raise ValueError(
            f"Final {col} missing count changed."
        )


print(
    "PASS: Final HIQ_E frequencies "
    "match original XPT/CDC."
)


# ============================================================
# 28. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 80)
print("HIQ_E CONVERSION COMPLETE")
print("=" * 80)


print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    "41475-51623"
)


print(
    "\nTiny artifacts:",
    tiny_total
)

print(
    "Tiny -> zero corrections:",
    0
)

print(
    "True source zeros:",
    true_zero_total
)

print(
    "Genuine decimal variables:",
    0
)


print(
    "\nCDC CHECK ITEMS not added:"
)

for item in CHECK_ITEMS:
    print(item)


print(
    "\nXPT -> CSV numeric differences:",
    total_numeric_differences
)

print(
    "Missing-position differences:",
    missing_mismatches
)

print(
    "Raw-format differences:",
    raw_mismatches
)

print(
    "Unwanted .0:",
    dot_zero_count
)

print(
    "Scientific notation:",
    scientific_count
)


print(
    "\nEXPECTED FINAL STATUS:"
)

print(
    "HIQ_E_corrected.csv = "
    "CORRECTED + FIDELITY CHECKED"
)

NHANES 2007-2008 HIQ_E XPT -> CSV
Rows: 10,149
Columns: 17
PASS: HIQ_E structure = 10,149 rows x 17 columns.
PASS: HIQ065, HIQ259 and HIQ269 correctly absent from released XPT.

Integer/code fields: 17
Genuine decimal fields: []

--- ORIGINAL XPT TINY VALUE CHECK ---
Tiny artifact occurrences: 0
PASS: HIQ_E contains NO tiny XPORT artifacts.

True source numeric zeros: 0
PASS: Original HIQ_E contains no numeric zeros.

PASS: SEQN range = 41475-51623.
PASS: Missing SEQN = 0.
PASS: Duplicate SEQN = 0.

--- FRACTIONAL VALUE CHECK ---
Genuine fractional variables: {}
PASS: Genuine decimal variables = 0.

--- CDC FREQUENCY CHECK ---
PASS: HIQ011
PASS: HIQ031A
PASS: HIQ031B
PASS: HIQ031C
PASS: HIQ031D
PASS: HIQ031E
PASS: HIQ031F
PASS: HIQ031G
PASS: HIQ031H
PASS: HIQ031I
PASS: HIQ031J
PASS: HIQ031AA
PASS: HIQ260
PASS: HIQ105
PASS: HIQ270
PASS: HIQ210

PASS: HIQ260 public-release distribution retained.
      Valid   = 248
      Missing = 9,901

PASS: Every released HIQ_E field contains whole-nu